# 03 Recommendation Engine

This notebook recommends three suitable OULAD modules for a student's next semester. The primary user is the student planning future study, with advisors using the same output during guidance conversations.


In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from data_loader import OUT_DIR, audit_and_clean_data, ensure_dirs, load_data
from features import build_weekly_base_features, make_enrollment_split
from recommendations import evaluate_recommenders
from scoring import apply_engagement_score, derive_engagement_weights

pd.set_option("display.max_columns", 120)
ensure_dirs()


## 1. Reuse The Behavioural History

The recommender should not be disconnected from Tasks 1 and 2. We reuse the weekly engagement features as student history: engagement score, material-study frequency, punctuality, and pre-start proactivity.


In [2]:
raw_data = load_data()
data, audits = audit_and_clean_data(raw_data)
weekly_base = build_weekly_base_features(data)
split = make_enrollment_split(weekly_base)
weights, weight_rationale = derive_engagement_weights(weekly_base, split)
weekly = apply_engagement_score(weekly_base, weights)
weekly.to_csv(OUT_DIR / "weekly_engagement_features.csv", index=False)

weekly[["code_module", "code_presentation", "id_student", "week", "engagement_score", "material_active_days", "punctuality_ratio", "pre_start_flag"]].head()


,code_module,code_presentation,id_student,week,engagement_score,material_active_days,punctuality_ratio,pre_start_flag
0,AAA,2013J,11391,1,63.7,4.0,1.0,1
1,AAA,2013J,11391,2,27.2,1.0,1.0,1
2,AAA,2013J,11391,3,40.2,2.0,1.0,1
3,AAA,2013J,11391,4,25.6,0.0,1.0,1
4,AAA,2013J,11391,5,45.4,2.0,1.0,1


## 2. Recommendation Methods

Two simple approaches are compared:

1. **Content-based:** recommend modules with strong historical success among students with similar academic profile and engagement history.
2. **Collaborative filtering:** use cosine similarity over prior successful module patterns, with a small boost for peers who showed pre-start proactivity.

Cosine similarity is appropriate here because the interaction vector is sparse and categorical: it compares the direction of successful module histories rather than raw counts.


In [3]:
recommender_metrics = evaluate_recommenders(data, weekly)
pd.DataFrame([recommender_metrics])


,holdout_students,content_hit_rate_at_3,cf_hit_rate_at_3,content_coverage,cf_coverage,catalog_modules,cold_start_strategy,similarity_metric,content_features
0,2479,0.110932,0.593788,5,7,7,"[AAA, EEE, GGG]",Cosine similarity over prior successful module...,"Highest education, age band, historical module..."


## 3. Holdout Evaluation

For returning students, the latest module is held out. The recommender succeeds if the actual next module appears in the top 3. This is a proxy evaluation, not a full production recommender benchmark, but it is enough to compare simple strategies for this assignment.


In [4]:
holdout_eval = pd.read_csv(OUT_DIR / "recommendation_holdout_eval.csv")
holdout_eval.head(10)


,id_student,actual_next_module,engagement_band,content_recs,cf_recs,content_hit_at_3,cf_hit_at_3
0,547267,BBB,low,"['AAA', 'GGG', 'EEE']","['AAA', 'GGG', 'EEE']",0,0
1,540568,BBB,medium,"['AAA', 'GGG', 'EEE']","['AAA', 'GGG', 'EEE']",0,0
2,540530,BBB,low,"['AAA', 'GGG', 'EEE']","['AAA', 'GGG', 'EEE']",0,0
3,538232,BBB,low,"['AAA', 'GGG', 'EEE']","['AAA', 'GGG', 'EEE']",0,0
4,528270,BBB,medium,"['AAA', 'GGG', 'EEE']","['AAA', 'GGG', 'EEE']",0,0
5,233771,EEE,high,"['AAA', 'GGG', 'EEE']","['AAA', 'BBB', 'CCC']",1,0
6,2367887,DDD,low,"['AAA', 'GGG', 'EEE']","['AAA', 'GGG', 'EEE']",0,0
7,602312,FFF,low,"['AAA', 'GGG', 'EEE']","['AAA', 'GGG', 'EEE']",0,0
8,2686609,FFF,low,"['AAA', 'GGG', 'EEE']","['AAA', 'GGG', 'EEE']",0,0
9,497278,GGG,low,"['AAA', 'GGG', 'EEE']","['AAA', 'GGG', 'EEE']",1,1


In [5]:
summary = pd.DataFrame({
    "approach": ["content_based", "collaborative_filtering"],
    "hit_at_3": [recommender_metrics["content_hit_rate_at_3"], recommender_metrics["cf_hit_rate_at_3"]],
    "coverage": [recommender_metrics["content_coverage"], recommender_metrics["cf_coverage"]],
    "catalog_modules": [recommender_metrics["catalog_modules"], recommender_metrics["catalog_modules"]],
})
summary


,approach,hit_at_3,coverage,catalog_modules
0,content_based,0.110932,5,7
1,collaborative_filtering,0.593788,7,7


## 4. Cold Start Strategy

For a brand new student with no behavioural history, collaborative filtering cannot be used. The implemented fallback recommends modules with strong historical pass rates, with a small preference for modules where proactive students historically performed well. This is simple, explainable, and avoids pretending we know personal preferences before observing any behaviour.


In [6]:
recommender_metrics["cold_start_strategy"]


['AAA', 'EEE', 'GGG']

## Decision

The collaborative method is expected to perform better for students with prior module history, while content-based recommendations provide a safer fallback for sparse-history or cold-start students. Both methods return exactly three modules and can be explained to a student or advisor without black-box language.
